In [ ]:
# ---
# jupyter:
#   jupytext:
#     formats: py:percent,ipynb
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.16.7
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---


In [2]:
import torch
import torch.nn.functional as F

# import helper functions
import sys, os

sys.path.append(os.getcwd())
from a01_helper import *

%load_ext autoreload
%autoreload 2
from a01_functions import train1



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 3 Backpropagation


In [56]:
# Let's fit the model with one hidden layer consisting of 50 units.
model = train1([50], nreps=1)
print("Training error:", F.mse_loss(y1, model(X1)).item())
print("Test error    :", F.mse_loss(y1test, model(X1test)).item())

# Extract parameters
pars = dict(model.named_parameters())
W1 = pars["0_weight"].data  # 1x50
b1 = pars["0_bias"].data  # 50
W2 = pars["1_weight"].data  # 50x1
b2 = pars["1_bias"].data  # 1


X1 shape: torch.Size([100, 1])
Repetition  0:          Current function value: 0.003361
         Iterations: 4639
         Function evaluations: 5069
         Gradient evaluations: 5060
best_cost=0.003
Training error: 0.003361145034432411
Test error    : 17.574420928955078


C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\site-packages\scipy\optimize\_minimize.py:779: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(fun, x0, args, jac, callback, **options)


## 3a Forward pass


In [57]:
# Compute results of forward pass on an example x (i.e., z1, z2, z3, z4, yhat, l) using Pytorch
x = X1test[1, :]
y = y1test[1, :]
print(f"x={x}, y={y}, yhat={model(x).detach()}, l={torch.nn.MSELoss()(y, model(x))}")


x=tensor([0.1030]), y=tensor([0.2253]), yhat=tensor([-0.9088]), l=1.2861924171447754


In [59]:
# Now do this by hand (including all intermediate values). You should get the same
# results as above.
#Ensuring Shape Consistency
W1 = W1.view(1,-1)
W2 = W2.view(-1,1)
b1 = b1.view(-1,1)
b2 = b2.view(1,1)

#Forward Pass
z1 = (W1.t() @ x).view(-1,1) #[50,1]
z2 = z1 + b1 #[50,1]
z3 = torch.sigmoid(z2).view(-1,1) #[50,1]
z4 = W2.t() @ z3 #[1,1] scalar
y_hat = z4 + b2
l = (y_hat-y)**2
print(x,y,y_hat,l)

tensor([0.1030]) tensor([0.2253]) tensor([[-0.9088]]) tensor([[1.2862]])


## 3b Backward pass


In [60]:
# Compute results of backward pass on example output (i.e., delta_x, delta_W1, delta_z1,
# delta_b1, delta_z2, delta_z3, delta_W2, delta_z4, delta_b2, delta_yhat, delta_l, delta_y)
##
delta_l = 1
delta_y = -2 * (y_hat-y) * delta_l
delta_yhat = 2 * (y_hat-y) * delta_l
delta_b2 = delta_yhat
delta_z4 = delta_yhat
delta_W2 = z3 * delta_z4
delta_z3 = W2 * delta_z4
delta_z2 = z3 * (1-z3) * delta_z3
delta_b1 = delta_z2.squeeze()
delta_z1 = delta_z2
delta_W1 = x.view(1,1) @ delta_z1.t()
delta_x = W1 @ delta_z1

In [61]:
# Use PyTorch's backprop
x.requires_grad = True
y.requires_grad = True
if x.grad is not None:
    x.grad.zero_()
if y.grad is not None:
    y.grad.zero_()
model.zero_grad()
t_yhat = model(x)
t_yhat.retain_grad()
t_l = torch.nn.MSELoss()(t_yhat, y)
t_l.backward()
t_delta_l = 1
t_delta_y = y.grad
t_delta_yhat = t_yhat.grad
t_delta_b2 = model.get_parameter("1_bias").grad
t_delta_W2 = model.get_parameter("1_weight").grad
t_delta_b1 = model.get_parameter("0_bias").grad
t_delta_W1 = model.get_parameter("0_weight").grad
t_delta_x = x.grad


In [62]:
# Check if equal (show squared error)
for v in ["y", "yhat", "b2", "W2", "b1", "W1", "x"]:
    pyt = eval("t_delta_" + v)
    you = torch.tensor(eval("delta_" + v))
    print(f"{v}, squared error={torch.sum((pyt - you) ** 2)}")


y, squared error=0.0
yhat, squared error=0.0
b2, squared error=0.0
W2, squared error=0.0
b1, squared error=2.4608201761033843e-13
W1, squared error=3.8441539990281826e-15
x, squared error=2.3283064365386963e-10


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16200\1916281833.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  you = torch.tensor(eval("delta_" + v))


In [46]:
# Check if equal (show actual values)
for v in ["l", "y", "yhat", "b2", "W2", "b1", "W1", "x"]:
    pyt = eval("t_delta_" + v)
    you = torch.tensor(eval("delta_" + v))
    print(f"{v}, pytorch={pyt}, you={you}")


l, pytorch=1, you=1
y, pytorch=tensor([-1.2442]), you=tensor([[-1.2442]])
yhat, pytorch=tensor([1.2442]), you=tensor([[1.2442]])
b2, pytorch=tensor([1.2442]), you=tensor([[1.2442]])
W2, pytorch=tensor([[1.2442e+00],
        [1.1665e+00],
        [1.1102e+00],
        [2.7942e-01],
        [1.4958e-01],
        [1.4573e-06],
        [3.3385e-01],
        [1.2442e+00],
        [3.2128e-04],
        [8.9588e-22],
        [3.5243e-05],
        [1.2442e+00],
        [2.9904e-11],
        [2.8355e-01],
        [1.5644e-12],
        [1.2442e+00],
        [1.7640e-29],
        [2.7213e-01],
        [1.2442e+00],
        [6.6820e-01],
        [3.2129e-09],
        [1.1169e+00],
        [1.2440e+00],
        [6.3251e-18],
        [9.4227e-01],
        [1.8315e-07],
        [9.3537e-22],
        [1.2442e+00],
        [1.7978e-07],
        [1.7749e-02],
        [1.2442e+00],
        [3.6152e-10],
        [1.8716e-02],
        [1.2437e+00],
        [2.7321e-07],
        [2.1552e-08],
        [3.429

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16200\3053622420.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  you = torch.tensor(eval("delta_" + v))
